# OCT-C8 Improved CNN-SE Training & Testing

This notebook trains the CNN-SE model from scratch with a consistent OCT image-quality preprocessing pipeline and evaluates validation/test images with the **same deterministic preprocessing**.

Key measures:
- Mild median denoising
- CLAHE contrast enhancement
- Aspect-ratio-preserving resize + padding
- Normalization
- Conservative OCT augmentation during training only
- CNN-SE attention blocks
- AdamW (with Adam fallback)
- Label smoothing
- Gradient clipping
- Early stopping and best-validation checkpoint
- Class-wise metrics, confusion matrix and macro ROC-AUC
- Memory-safe `tf.data` streaming
- Single-image and folder prediction code using the exact test preprocessing

**Important:** validation and test images are never augmented, and the test set is not used for model selection.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("junituleislamriad/retinal-oct-image-classification-c8")

In [ ]:
path = "/kaggle/input/datasets/junituleislamriad/retinal-oct-image-classification-c8"

In [ ]:
import os
from pathlib import Path

print("Dataset downloaded to:")
print(path)

print("\nContents:")
for item in os.listdir(path):
    print(item)

In [ ]:
from pathlib import Path

dataset_path = Path(path)

for p in dataset_path.iterdir():
    print(p)

In [ ]:
from pathlib import Path

root = Path(path)

train_candidates = list(root.rglob("train"))
val_candidates   = list(root.rglob("val"))
test_candidates  = list(root.rglob("test"))

print("TRAIN candidates:")
for p in train_candidates:
    print(p)

print("\nVAL candidates:")
for p in val_candidates:
    print(p)

print("\nTEST candidates:")
for p in test_candidates:
    print(p)

In [ ]:
import os

# Kaggle working directory
BASE_DIR = "/kaggle/working/OCT_C8_CNN_SE_Results"

# Create folders for permanent research outputs
for folder in ["models", "results", "figures", "logs"]:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Kaggle results storage created:")
print(BASE_DIR)

# Verify
for folder in ["models", "results", "figures", "logs"]:
    path = os.path.join(BASE_DIR, folder)
    print(f"{folder}: {path}")

In [ ]:
import os, gc, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, LearningRateScheduler

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ===================== COLAB PATH CONFIG =====================

from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/junituleislamriad/retinal-oct-image-classification-c8/RetinalOCT_Dataset")

def find_folder(root, folder_name):
    matches = [
        p for p in root.rglob(folder_name)
        if p.is_dir()
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find '{folder_name}' inside {root}"
        )

    # Prefer a folder that contains the expected classes
    expected = {
        "AMD", "CNV", "CSR", "DME",
        "DR", "DRUSEN", "MH", "NORMAL"
    }

    for p in matches:
        children = {x.name.upper() for x in p.iterdir() if x.is_dir()}
        if expected.issubset(children):
            return p

    return matches[0]

TRAIN_DIR = find_folder(DATA_ROOT, "train")
VAL_DIR   = find_folder(DATA_ROOT, "val")
TEST_DIR  = find_folder(DATA_ROOT, "test")

OUTPUT_DIR = Path(BASE_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "AMD", "CNV", "CSR", "DME",
    "DR", "DRUSEN", "MH", "NORMAL"
]

NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 60

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
DROPOUT = 0.30
SE_REDUCTION = 4

DENOISE_KERNEL = 3
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)

print("TRAIN:", TRAIN_DIR)
print("VAL  :", VAL_DIR)
print("TEST :", TEST_DIR)
print("OUTPUT:", OUTPUT_DIR)

In [ ]:
for name, folder in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR)
]:
    print("\n" + "="*50)
    print(name)

    for cls in CLASS_NAMES:
        class_dir = folder / cls

        if class_dir.exists():
            count = len([
                p for p in class_dir.rglob("*")
                if p.is_file()
            ])
            print(f"{cls:8s}: {count}")
        else:
            print(f"{cls:8s}: MISSING")

In [ ]:
from pathlib import Path

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def list_images(folder):
    folder = Path(folder)
    return sorted([
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS
    ])

print("list_images function is ready.")

## OCT image preprocessing

The deterministic preprocessing below is applied to **train, validation, test, and new images**:
1. grayscale loading
2. mild median denoising
3. CLAHE
4. aspect-ratio-preserving resize with padding
5. scaling to `[0,1]`
6. grayscale replicated to 3 channels for the CNN

Only training data receive the separate mild augmentation step.

In [ ]:
# ===================== PREPROCESSING =====================

def preprocess_oct(path, img_size=IMG_SIZE, augment=False):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {path}")

    # Mild denoising; avoid aggressive smoothing of retinal structures.
    img = cv2.medianBlur(img, DENOISE_KERNEL)

    # Mild local contrast enhancement.
    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP,
        tileGridSize=CLAHE_GRID
    )
    img = clahe.apply(img)

    # Preserve aspect ratio and pad.
    h, w = img.shape
    th, tw = img_size
    scale = min(tw / w, th / h)
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))

    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((th, tw), dtype=np.uint8)
    y0 = (th - nh) // 2
    x0 = (tw - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = img

    img = canvas.astype(np.float32) / 255.0

    if augment:
        angle = np.random.uniform(-5, 5)
        tx = np.random.uniform(-0.03, 0.03) * tw
        ty = np.random.uniform(-0.03, 0.03) * th

        M = cv2.getRotationMatrix2D((tw/2, th/2), angle, 1.0)
        M[0, 2] += tx
        M[1, 2] += ty

        img = cv2.warpAffine(
            img, M, (tw, th),
            borderMode=cv2.BORDER_REFLECT_101
        )

        if np.random.rand() < 0.35:
            alpha = np.random.uniform(0.90, 1.10)
            beta = np.random.uniform(-0.04, 0.04)
            img = np.clip(img * alpha + beta, 0, 1)

        if np.random.rand() < 0.25:
            zoom = np.random.uniform(0.97, 1.03)
            ch = max(1, int(th / zoom))
            cw = max(1, int(tw / zoom))
            y = (th - ch) // 2
            x = (tw - cw) // 2
            crop = img[y:y+ch, x:x+cw]
            img = cv2.resize(crop, (tw, th), interpolation=cv2.INTER_LINEAR)

    return np.repeat(img[..., None], 3, axis=-1).astype(np.float32)


# Preview one image after preprocessing.
sample_paths = list_images(TRAIN_DIR)
if sample_paths:
    sample = preprocess_oct(sample_paths[0], augment=False)
    plt.figure(figsize=(7,5))
    plt.imshow(sample[...,0], cmap="gray")
    plt.axis("off")
    plt.title("Preprocessed OCT example")
    plt.show()

In [ ]:
# ===================== BUILD PATH TABLES =====================

def make_records(folder):
    records = []
    for idx, c in enumerate(CLASS_NAMES):
        d = Path(folder) / c
        if not d.exists():
            raise FileNotFoundError(f"Missing class folder: {d}")
        for p in list_images(d):
            records.append({"path": str(p), "label": idx, "class": c})

    if not records:
        raise RuntimeError(f"No images found in {folder}")
    return pd.DataFrame(records)

train_df = make_records(TRAIN_DIR)
val_df   = make_records(VAL_DIR)
test_df  = make_records(TEST_DIR)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

display(train_df.groupby("class").size().reindex(CLASS_NAMES).to_frame("train"))
display(val_df.groupby("class").size().reindex(CLASS_NAMES).to_frame("val"))
display(test_df.groupby("class").size().reindex(CLASS_NAMES).to_frame("test"))

In [ ]:
# ===================== MEMORY-SAFE TF.DATA =====================

AUTOTUNE = tf.data.AUTOTUNE

def _py_preprocess(path_bytes, training):
    return preprocess_oct(path_bytes.decode("utf-8"), augment=training)

def tf_load(path, label, training=False):
    x = tf.numpy_function(
        lambda p: _py_preprocess(p, training),
        [path],
        tf.float32
    )
    x.set_shape((*IMG_SIZE, 3))
    y = tf.one_hot(label, NUM_CLASSES)
    return x, y

def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values, df["label"].values)
    )

    if training:
        ds = ds.shuffle(
            min(len(df), 10000),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda p, y: tf_load(p, y, training),
        num_parallel_calls=AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, True)
val_ds   = make_dataset(val_df, False)
test_ds  = make_dataset(test_df, False)

print("Datasets created. Images are streamed batch-by-batch; the full dataset is not loaded into RAM.")

In [ ]:
# ===================== CNN-SE MODEL =====================

def se_block(x, reduction=SE_REDUCTION, name="se"):
    channels = int(x.shape[-1])
    s = layers.GlobalAveragePooling2D(name=name+"_gap")(x)
    s = layers.Dense(max(channels // reduction, 4),
                     activation="relu",
                     name=name+"_fc1")(s)
    s = layers.Dense(channels,
                     activation="sigmoid",
                     name=name+"_fc2")(s)
    s = layers.Reshape((1,1,channels), name=name+"_reshape")(s)
    return layers.Multiply(name=name+"_scale")([x, s])

def conv_block(x, filters, drop=0.0, name="block"):
    x = layers.Conv2D(filters, 3, padding="same",
                     use_bias=False, name=name+"_conv1")(x)
    x = layers.BatchNormalization(name=name+"_bn1")(x)
    x = layers.Activation("relu", name=name+"_relu1")(x)

    x = layers.Conv2D(filters, 3, padding="same",
                     use_bias=False, name=name+"_conv2")(x)
    x = layers.BatchNormalization(name=name+"_bn2")(x)
    x = layers.Activation("relu", name=name+"_relu2")(x)

    x = se_block(x, name=name+"_se")
    x = layers.MaxPooling2D(2, name=name+"_pool")(x)

    if drop:
        x = layers.Dropout(drop, name=name+"_drop")(x)
    return x

def build_cnn_se():
    inp = layers.Input((*IMG_SIZE, 3), name="input")

    x = layers.Conv2D(32, 3, padding="same",
                     use_bias=False, name="stem_conv")(inp)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    x = conv_block(x, 32, 0.05, "block1")
    x = conv_block(x, 64, 0.10, "block2")
    x = conv_block(x, 128, 0.15, "block3")
    x = conv_block(x, 256, 0.20, "block4")

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)
    x = layers.BatchNormalization(name="feature_bn")(x)
    x = layers.Dropout(DROPOUT, name="classifier_dropout")(x)

    out = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        kernel_regularizer=regularizers.l2(1e-5),
        name="classifier"
    )(x)

    return models.Model(inp, out, name="CNN_SE_OCT8")

model = build_cnn_se()
model.summary()

In [ ]:
# ===================== COMPILE =====================

try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0
    )
    print("Using AdamW")
except Exception:
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    )
    print("AdamW unavailable; using Adam")

loss_fn = tf.keras.losses.CategoricalCrossentropy(
    label_smoothing=LABEL_SMOOTHING
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy"]
)

model.save_weights(
    str(OUTPUT_DIR / "initial_CNN_SE_weights.weights.h5")
)

In [ ]:
# ===================== CALLBACKS =====================

best_path = OUTPUT_DIR / "best_CNN_SE_OCT8.keras"
log_path = OUTPUT_DIR / "training_log.csv"

# Cosine decay implemented as a Keras schedule.
steps_per_epoch = int(np.ceil(len(train_df) / BATCH_SIZE))
total_steps = max(1, steps_per_epoch * EPOCHS)

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=total_steps,
    alpha=0.01
)

# Rebuild optimizer with schedule.
try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY,
        clipnorm=1.0
    )
except Exception:
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=lr_schedule,
        clipnorm=1.0
    )

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy"]
)

callbacks = [
    ModelCheckpoint(
        str(best_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger(str(log_path), append=False)
]

print("Best model:", best_path)
print("Training log:", log_path)

In [ ]:
# ===================== TRAIN FROM SCRATCH =====================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

model.save(str(OUTPUT_DIR / "final_CNN_SE_OCT8.keras"))
print("Training completed.")

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(LOG_DIR, "training_history.csv"),
    index=False
)

print("Training history saved.")

In [ ]:
history_df = pd.read_csv(
    "/kaggle/working/OCT_C8_CNN_SE_Results/logs/training_history.csv"
)

print(history_df.head())

In [ ]:
# ===================== TRAINING CURVES =====================
import pandas as pd
hist = pd.DataFrame(history.history)

plt.figure(figsize=(8,5))
plt.plot(hist["accuracy"], label="Train")
plt.plot(hist["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN-SE Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist["loss"], label="Train")
plt.plot(hist["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN-SE Loss")
plt.legend()
plt.show()

best_epoch = int(hist["val_accuracy"].idxmax()) + 1
print("Best validation epoch:", best_epoch)
print("Best validation accuracy:", float(hist["val_accuracy"].max()))

In [ ]:
# ===================== EVALUATION =====================

def evaluate_dataset(model, df, ds, name):
    probs = model.predict(ds, verbose=1)
    y_true = df["label"].values
    y_pred = np.argmax(probs, axis=1)

    print("\n" + "="*70)
    print(name)
    print("="*70)

    print(classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(
        y_true, y_pred,
        labels=np.arange(NUM_CLASSES)
    )

    accuracy = float(np.mean(y_true == y_pred))
    print("Accuracy:", f"{accuracy:.4f}")

    try:
        y_bin = label_binarize(
            y_true,
            classes=np.arange(NUM_CLASSES)
        )
        auc = roc_auc_score(
            y_bin,
            probs,
            multi_class="ovr",
            average="macro"
        )
        print("Macro ROC-AUC:", f"{auc:.4f}")
    except Exception as e:
        auc = np.nan
        print("ROC-AUC unavailable:", e)

    plt.figure(figsize=(8,7))
    plt.imshow(cm)
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
    plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
    plt.colorbar()

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, str(cm[i,j]),
                     ha="center", va="center")
    plt.tight_layout()
    plt.show()

    return probs, y_true, y_pred, cm, auc

In [ ]:
import tensorflow as tf
import os

MODEL_PATH = "/kaggle/working/OCT_C8_CNN_SE_Results/best_CNN_SE_OCT8.keras"

print("Model exists:", os.path.exists(MODEL_PATH))

model = tf.keras.models.load_model(MODEL_PATH)

print("\nModel loaded successfully!")
model.summary()

In [ ]:
# ===================== VALIDATION =====================

val_probs, val_true, val_pred, val_cm, val_auc = evaluate_dataset(
    model, val_df, val_ds, "OCT-C8 VALIDATION"
)

In [ ]:
# ===================== FINAL TEST =====================

# This is the first time the untouched OCT-C8 test set is used.
test_probs, test_true, test_pred, test_cm, test_auc = evaluate_dataset(
    model, test_df, test_ds, "OCT-C8 TEST"
)

pd.DataFrame(
    classification_report(
        test_true,
        test_pred,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )
).T.to_csv(
    OUTPUT_DIR / "OCT_C8_test_classification_report.csv"
)

pd.DataFrame(
    test_cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    OUTPUT_DIR / "OCT_C8_test_confusion_matrix.csv"
)

np.save(
    OUTPUT_DIR / "OCT_C8_test_probabilities.npy",
    test_probs
)

print("Test results saved to:", OUTPUT_DIR)

## New-image testing

For a new OCT image, **augmentation is disabled**. The same denoising, CLAHE, resize/padding and normalization used for validation/test is applied before prediction.

In [ ]:
# ===================== SINGLE IMAGE TEST =====================

def predict_single_image(model, image_path, show=True):
    image_path = Path(image_path)

    x = preprocess_oct(
        image_path,
        augment=False
    )

    probs = model.predict(
        x[None, ...],
        verbose=0
    )[0]

    idx = int(np.argmax(probs))

    if show:
        plt.figure(figsize=(7,5))
        plt.imshow(x[...,0], cmap="gray")
        plt.axis("off")
        plt.title(
            f"Predicted: {CLASS_NAMES[idx]} | "
            f"Confidence: {probs[idx]*100:.2f}%"
        )
        plt.show()

        top = np.argsort(probs)[::-1]
        print("Top predictions:")
        for k in top[:3]:
            print(
                f"{CLASS_NAMES[k]:8s}: "
                f"{probs[k]*100:.2f}%"
            )

    return CLASS_NAMES[idx], float(probs[idx]), probs

# Example:
# NEW_IMAGE = r"E:\Phd\datasets\OCT\sample.jpg"
# predict_single_image(model, NEW_IMAGE)

In [ ]:
# ===================== FOLDER TEST =====================

def predict_folder(model, folder, output_csv=None):
    paths = list_images(Path(folder))
    if not paths:
        raise RuntimeError(f"No images found in {folder}")

    rows = []

    for p in paths:
        try:
            x = preprocess_oct(
                p,
                augment=False
            )

            probs = model.predict(
                x[None, ...],
                verbose=0
            )[0]

            idx = int(np.argmax(probs))

            row = {
                "path": str(p),
                "predicted_class": CLASS_NAMES[idx],
                "confidence": float(probs[idx])
            }

            for i, c in enumerate(CLASS_NAMES):
                row[f"prob_{c}"] = float(probs[i])

            rows.append(row)

        except Exception as e:
            rows.append({
                "path": str(p),
                "error": str(e)
            })

    result = pd.DataFrame(rows)

    if output_csv is not None:
        result.to_csv(output_csv, index=False)

    return result

# Example:
# result = predict_folder(
#     model,
#     r"E:\Phd\OCT_new_images",
#     OUTPUT_DIR / "new_images_predictions.csv"
# )
# display(result.head())

## Scientific safeguards

- No OCT-C8 test image is used for training.
- Validation is used only for model selection/early stopping.
- Test preprocessing is identical to validation preprocessing and has no augmentation.
- The full dataset is streamed through `tf.data`; it is not stacked into one giant NumPy array.
- For strong paper claims, repeat the final experiment across multiple random seeds and report mean ± SD and confidence intervals where appropriate.
- Preprocessing itself should be validated experimentally: denoising/CLAHE can help, but excessive enhancement can remove clinically meaningful information.

In [ ]:
import os

path = "/content/OCT_C8_CNN_SE_Improved"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Contents:")
    print(os.listdir(path))

In [ ]:
!git add .
!git commit -m 
!git push